# 03 — Chatbot RAG Interactivo

MVP completo del chatbot. Requiere que `02_indexacion_vectorstore.ipynb` haya sido ejecutado (ChromaDB en `../chroma_db/`) y que Ollama esté corriendo con el modelo descargado.

**Prerequisito:** `ollama pull mistral:7b-instruct`

In [ ]:
import sys
sys.path.insert(0, '..')

from src.rag_chain import construir_cadena, formatear_respuesta

# Cambiar el modelo aquí si se quiere probar otro
MODELO = 'mistral:7b-instruct'  # o 'llama3.1:8b', 'gemma2:9b'

print(f'Cargando cadena RAG con modelo: {MODELO}')
chain = construir_cadena(model=MODELO, k=3, chroma_dir='../chroma_db')
print('Listo.')

In [ ]:
def preguntar(pregunta: str, mostrar_fuentes: bool = True):
    print(f'Pregunta: {pregunta}')
    print('-' * 60)
    resultado = chain(pregunta)
    print(resultado['result'])
    if mostrar_fuentes:
        print()
        fuentes = {(d.metadata['nombre_doc'], d.metadata['seccion'])
                   for d in resultado.get('source_documents', [])}
        print('Fuentes consultadas:')
        for nombre, seccion in sorted(fuentes):
            print(f'  • {nombre} — {seccion[:70]}')
    print()

In [ ]:
# Prueba 1: Producto defectuoso
preguntar('¿Cuáles son mis derechos si un producto que compré está defectuoso?')

In [ ]:
# Prueba 2: Reclamo ante INDECOPI
preguntar('¿Cómo presento una queja ante INDECOPI?')

In [ ]:
# Prueba 3: SOAT
preguntar('¿Qué cubre el SOAT en caso de accidente de tránsito?')

In [ ]:
# Prueba 4: Atención preferente
preguntar('¿Tengo derecho a atención preferente si soy adulto mayor?')

In [ ]:
# Evaluación con ROUGE-L (comparar contra respuesta de referencia manual)
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

# Definir pares (pregunta, respuesta de referencia simplificada manualmente)
pares_evaluacion = [
    (
        '¿Qué es el libro de reclamaciones?',
        'El libro de reclamaciones es un registro donde puedes dejar constancia de '
        'tu queja o reclamo sobre un producto o servicio. Todo establecimiento que '
        'vende bienes o presta servicios está obligado a tenerlo y a dártelo cuando lo pidas.',
    ),
    (
        '¿Cuándo vence la garantía de un producto?',
        'La garantía legal de un producto es de 2 años desde que lo compraste. '
        'En ese plazo, si el producto tiene defectos de fabricación, el vendedor '
        'debe repararlo, cambiarlo o devolverte el dinero.',
    ),
]

print('Evaluación ROUGE-L (0 a 1, mayor es mejor)\n')
for pregunta, referencia in pares_evaluacion:
    resultado = chain(pregunta)
    respuesta = resultado['result']
    scores = scorer.score(referencia, respuesta)
    print(f'Pregunta: {pregunta}')
    print(f'ROUGE-L F1: {scores["rougeL"].fmeasure:.3f}')
    print()

In [ ]:
# Chat interactivo en el notebook
print('Chatbot activado. Escribe \'salir\' para terminar.\n')
while True:
    pregunta = input('Tú: ').strip()
    if pregunta.lower() in ('salir', 'exit', 'quit', ''):
        break
    resultado = chain(pregunta)
    print(f'\nAsistente: {resultado["result"]}\n')